# Phase 1 — Swin Transformer V2 Training (Kaggle)

Self-contained notebook that trains the Swin V2 classifier on the ISIC 2019 dataset (binary: melanoma vs. not_melanoma).

**Outputs:**
- `swin_best.pth` — best model weights (download → place in `checkpoints/` locally)
- `training_log.csv` — per-epoch metrics

**Runtime:** Kaggle T4 GPU (16GB VRAM), ~2-3 hours for 50 epochs

## 1. Setup

In [ ]:
!pip install -q timm albumentations==2.0.8

In [ ]:
import os
import csv
import time
from pathlib import Path
from typing import List, Tuple, Optional, Callable, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# ======================== EDIT THESE PATHS ========================
# If using Kaggle dataset, paths are typically /kaggle/input/...
ISIC2019_IMAGES = "/kaggle/input/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
ISIC2019_CSV = "/kaggle/input/isic-2019/ISIC_2019_Training_GroundTruth.csv"
# ==================================================================

# Hyperparameters
MODEL_NAME = "swinv2_base_window12to16_192to256"
IMG_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 50
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.1
EARLY_STOPPING_PATIENCE = 7

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Load & Preprocess ISIC 2019 Labels

ISIC 2019 has 8 disease categories. We convert to binary:
- **MEL** → melanoma (1)
- **NV, BCC, AK, BKL, DF, VASC, SCC** → not_melanoma (0)

In [ ]:
df = pd.read_csv(ISIC2019_CSV)
print(f"Total samples: {len(df)}")

# Determine original class
class_cols = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
class_cols = [c for c in class_cols if c in df.columns]
df["original_class"] = df[class_cols].idxmax(axis=1)

# Binary label: MEL=1, everything else=0
df["label"] = (df["original_class"] == "MEL").astype(int)

print("\n--- Original Distribution ---")
for cls in class_cols:
    count = (df["original_class"] == cls).sum()
    print(f"  {cls:>5s}: {count:>6d}  →  {'melanoma' if cls == 'MEL' else 'not_melanoma'}")

print(f"\n--- Binary Distribution ---")
print(f"  Melanoma: {df['label'].sum():>6d} ({df['label'].mean()*100:.1f}%)")
print(f"  Not Melanoma:   {(df['label']==0).sum():>6d} ({(1-df['label'].mean())*100:.1f}%)")

In [ ]:
# Verify images exist
img_dir = Path(ISIC2019_IMAGES)
valid_images = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Verifying images"):
    for ext in [".jpg", ".jpeg", ".png"]:
        path = img_dir / f"{row['image']}{ext}"
        if path.exists():
            valid_images.append({"image": row["image"], "path": str(path), "label": row["label"]})
            break

df_valid = pd.DataFrame(valid_images)
print(f"\nValid images: {len(df_valid)} / {len(df)}")

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(df_valid, test_size=0.30, stratify=df_valid["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)} (mel: {train_df['label'].sum()})")
print(f"Val:   {len(val_df)} (mel: {val_df['label'].sum()})")
print(f"Test:  {len(test_df)} (mel: {test_df['label'].sum()})")

# Save splits for reproducibility
train_df.to_csv(OUTPUT_DIR / "train_split.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val_split.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_split.csv", index=False)

## 4. Dataset & Transforms

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def get_transforms(split: str, img_size: int = 256):
    if split == "train":
        return A.Compose([
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Transpose(p=0.3),
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                A.CLAHE(clip_limit=2.0, p=1.0),
            ], p=0.3),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=8, p=0.2),
            A.CoarseDropout(max_holes=4, max_height=int(img_size*0.08), max_width=int(img_size*0.08), p=0.2),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(height=img_size, width=img_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])


class ISICDataset(Dataset):
    """Dataset that reads from a DataFrame with 'path' and 'label' columns."""
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = np.array(Image.open(row["path"]).convert("RGB"))
        label = int(row["label"])
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, label

In [ ]:
# Create datasets
train_ds = ISICDataset(train_df, get_transforms("train", IMG_SIZE))
val_ds = ISICDataset(val_df, get_transforms("val", IMG_SIZE))

# Weighted sampler for class imbalance
labels = train_df["label"].values
class_counts = np.bincount(labels)
class_weights = len(labels) / (len(class_counts) * class_counts)
sample_weights = [class_weights[l] for l in labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

print(f"Class weights: not_melanoma={class_weights[0]:.3f}, melanoma={class_weights[1]:.3f}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 5. Model

In [ ]:
class SwinV2Classifier(nn.Module):
    def __init__(self, model_name, num_classes=2, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.embed_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.embed_dim, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def get_backbone_params(self):
        return self.backbone.parameters()

    def get_head_params(self):
        return self.head.parameters()


model = SwinV2Classifier(MODEL_NAME, num_classes=2, pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")

## 6. Training Loop

In [ ]:
# Loss, optimizer, scheduler
weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW([
    {"params": model.get_backbone_params(), "lr": LR_BACKBONE},
    {"params": model.get_head_params(), "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = GradScaler()

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return {"loss": running_loss/total, "accuracy": correct/total}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(device_type=device.type, dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)
        probs = torch.softmax(logits.float(), dim=1)
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        all_probs.append(probs[:, 1].cpu())
        all_labels.append(labels.cpu())
    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    try:
        auroc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auroc = 0.0
    return {"loss": running_loss/total, "accuracy": correct/total, "auroc": auroc}

In [ ]:
# Training loop with early stopping
log_path = OUTPUT_DIR / "training_log.csv"
log_file = open(log_path, "w", newline="")
csv_writer = csv.DictWriter(log_file, fieldnames=["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "val_auroc", "lr", "time_s"])
csv_writer.writeheader()

best_auroc = 0.0
patience_counter = 0

print(f"Training for {MAX_EPOCHS} epochs...\n")

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    train_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_m = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    elapsed = time.time() - t0

    row = {
        "epoch": epoch, "train_loss": f"{train_m['loss']:.4f}", "train_acc": f"{train_m['accuracy']:.4f}",
        "val_loss": f"{val_m['loss']:.4f}", "val_acc": f"{val_m['accuracy']:.4f}",
        "val_auroc": f"{val_m['auroc']:.4f}", "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        "time_s": f"{elapsed:.1f}",
    }
    csv_writer.writerow(row)
    log_file.flush()

    print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | Train Loss: {train_m['loss']:.4f} Acc: {train_m['accuracy']:.4f} | "
          f"Val Loss: {val_m['loss']:.4f} Acc: {val_m['accuracy']:.4f} AUROC: {val_m['auroc']:.4f} | {elapsed:.1f}s")

    if val_m["auroc"] > best_auroc:
        best_auroc = val_m["auroc"]
        patience_counter = 0
        torch.save(model.state_dict(), OUTPUT_DIR / "swin_best.pth")
        print(f"  ✓ New best AUROC: {best_auroc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

log_file.close()
print(f"\nDone! Best AUROC: {best_auroc:.4f}")
print(f"Checkpoint: {OUTPUT_DIR / 'swin_best.pth'}")

## 7. Quick Test Evaluation

In [ ]:
# Load best and evaluate on test set
model.load_state_dict(torch.load(OUTPUT_DIR / "swin_best.pth", weights_only=True))
test_ds = ISICDataset(test_df, get_transforms("val", IMG_SIZE))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

test_m = validate(model, test_loader, criterion, DEVICE)
print(f"\n--- Test Results ---")
print(f"  Accuracy: {test_m['accuracy']:.4f}")
print(f"  AUROC:    {test_m['auroc']:.4f}")

## 8. Download Instructions

After training:
1. Download `swin_best.pth` from the Output panel
2. Place it in `c:\SCD-II\checkpoints\swin_best.pth` on your local machine
3. Run `python main.py diagnose path/to/image.jpg` locally